#Data Reading

## Common dbutils Modules

### 1. dbutils.fs (File System Utilities)
Used to work with DBFS (Databricks File System).

**List files:**
```python
dbutils.fs.ls("/FileStore/")
```

**Create directory:**
```python
dbutils.fs.mkdirs("/FileStore/test")
```

**Copy files:**
```python
dbutils.fs.cp(
    "dbfs:/FileStore/source.csv",
    "dbfs:/FileStore/target.csv"
)
```

**Remove files:**
```python
dbutils.fs.rm("/FileStore/test", recurse=True)
```

---

### 2. dbutils.secrets (Secret Management)
Used to securely access passwords, API keys, and tokens.

**List secret scopes:**
```python
dbutils.secrets.listScopes()
```

**Get secret value:**
```python
dbutils.secrets.get(scope="my-scope", key="api-key")
```


In [0]:
dbutils.fs.ls('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/')

##Data Reading CSV

In [0]:
df_csv = spark.read.format('csv').option('inferschema', True).option('header', True).load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')

In [0]:
df_csv.show()
df_csv.display()

##Data Reading JSON

In [0]:
df_json = spark.read.format('json').option('inferschema', True)\
            .option('header',True)\
            .option('multiline', False)\
            .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/drivers.json')

In [0]:
df_json.show()
df_json.display()

#Schema Definition

In [0]:
df_csv.printSchema()
df_json.printSchema()



We can define schema to the table or data Frame in pyspark using the DDL Schmema or the Struct Type Schema




##DDL SCHEMA

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df_csv = spark.read.format('csv')\
                    .option('inferschema', True)\
                    .option('header', True)\
                    .schema(my_ddl_schema)\
                    .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')

In [0]:
df_csv.printSchema()

##StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *  

In [0]:
my_struct_csv_schema  = StructType([ StructField('Item_Identifier',StringType(),True), StructField('Item_Weight',StringType(),True), StructField('Item_Fat_Content',StringType(),True), StructField('Item_Visibility',StringType(),True), StructField('Item_MRP',StringType(),True), StructField('Outlet_Identifier',StringType(),True), StructField('Outlet_Establishment_Year',StringType(),True), StructField('Outlet_Size',StringType(),True), StructField('Outlet_Location_Type',StringType(),True), StructField('Outlet_Type',StringType(),True), StructField('Item_Outlet_Sales',StringType(),True)
])


In [0]:
df_my_struct_csv = spark.read.format('csv')\
                        .option('infershema', True)\
                        .option('header', True)\
                        .schema(my_struct_csv_schema)\
                        .load('/Volumes/pyspark_learnings_db/learning/staging_pyspark_learning/BigMart Sales.csv')


In [0]:
df_my_struct_csv.printSchema()

#TRANSFORMATIONS

In [0]:
df_csv.printSchema()

##SELECT

In [0]:
df_csv.printSchema()

## Select Column Methods

**Two ways to select columns in PySpark:**

1. **String-based selection** - `select("col1", "col2")`
   * Uses string column names
   * Suitable for simple column selection
   * Quick and concise

2. **Column object-based selection** - `select(col("col1"), col("col2"))`
   * Uses PySpark Column objects
   * Allows additional operations:
     * Aliasing (`.alias()`)
     * Expressions and calculations
     * Casting (`.cast()`)
     * Transformations

**Note:** For simple selection, both produce the same result.

**Example with Column objects:**
```python
from pyspark.sql.functions import col

df_csv.select(
    col("Item_Identifier").alias("ID"),
    col("Item_Weight") * 2
).show()
```


In [0]:
df_csv.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

In [0]:
df_csv.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

##ALIAS

In [0]:
df_csv.select(col('Item_Identifier').alias('Item_ID')).display()

###FILTER

In [0]:
df_csv.display()

![image_1787641808078.png](./image_1787641808078.png "image_1787641808078.png")

###Scenario - 1


In [0]:
df_csv.filter((col('Item_Fat_Content') == 'Regular')).display()

###Scenario - 2

In [0]:
df_csv.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') <10 ) ).display()

###Scenarion - 3

# PySpark Quick Reference: `isNull()` and `isin()`

## 🔍 Null Checking

**Correct way to check for NULL values:**

```python
col("column_name").isNull()      # Finds null values
col("column_name").isNotNull()   # Finds non-null values
```

**❌ Common mistakes (do NOT use):**

```python
col("column_name") == NULL  # Wrong!
col("column_name") == None  # Wrong!
```

> **Why?** Normal equality comparison (`==`) cannot correctly identify SQL NULL values.

---

## 📊 Value Matching

### Checking a Single Value

```python
col("city") == "Chennai"
```

### Checking Multiple Possible Values

**Using `isin()` (recommended):**

```python
col("city").isin("Chennai", "Bangalore")
```

**This is equivalent to:**

```python
(col("city") == "Chennai") | (col("city") == "Bangalore")
```

> **Key Point:** `isin()` uses **OR** logic because a column can match **any one** of the provided values.

---

## ⚠️ Common Mistake: Using AND Instead of OR

**❌ This is INCORRECT:**

```python
(col("city") == "Chennai") & (col("city") == "Bangalore")
```

> **Why it fails:** This returns **no rows** because one column value cannot be both "Chennai" **and** "Bangalore" at the same time.

---

## 🚫 Excluding Multiple Values

```python
~col("city").isin("Chennai", "Bangalore")
```

**Equivalent to:**

```python
(col("city") != "Chennai") & (col("city") != "Bangalore")
```

---

## 🔧 Logical Operators

| Operator | Meaning | Example |
|----------|---------|----------|
| `&` | AND | `(condition1) & (condition2)` |
| `\|` | OR | `(condition1) \| (condition2)` |
| `~` | NOT | `~condition` |

> **Important:** Always put each PySpark condition inside **parentheses**.

---

## 💡 Easy Rules to Remember

| Function | Use Case |
|----------|----------|
| `isNull()` | Check for NULL values |
| `==` | Check for one specific value |
| `isin()` | Check if value matches **any** from multiple options |

---

## 🎯 Key Takeaway

**`isin("value1", "value2")` is equivalent to equality conditions joined using `\|` (OR), not `&` (AND).**

In [0]:
df_csv.filter((col('Outlet_Location_Type').isin('Tier 1', 'Tier 2')) & (col('Outlet_Size').isNull())).display()